In [1]:
import pandas as pd
from datetime import datetime
import sys
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, r2_score
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
pio.templates.default = "plotly_white"
# --- Statistical Analysis Libraries ---
from scipy import stats
from scipy.stats import normaltest, shapiro, anderson
from scipy.stats import pearsonr, spearmanr, kendalltau
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.feature_selection import (
    SelectKBest, f_regression, mutual_info_regression,
    RFE, RFECV, SelectFromModel
)
# Models
from sklearn.linear_model import (
    LinearRegression, Ridge, Lasso, ElasticNet,
    HuberRegressor, RANSACRegressor, TheilSenRegressor
)
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor, ExtraTreesRegressor,
    GradientBoostingRegressor, AdaBoostRegressor,
    VotingRegressor, StackingRegressor, BaggingRegressor
)
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor

# Metrics
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    mean_absolute_percentage_error, explained_variance_score,
    max_error, median_absolute_error
)


RANDOM_STATE = 42

In [2]:
data_path = "..\dataset\insurance.csv"
df = pd.read_csv(data_path)
print("Dataset loaded successfully!")

Dataset loaded successfully!


In [3]:
class AdvancedFeatureEngineer:
    """
    Comprehensive feature engineering pipeline with domain knowledge
    """
    
    def __init__(self, verbose=True):
        self.verbose = verbose
        self.feature_names = []
        self.original_features = []
        
    def fit_transform(self, df):
        """
        Create all engineered features
        """
        if self.verbose:
            print("🔧 Starting Feature Engineering Pipeline...")
        
        df_feat = df.copy()
        self.original_features = df.columns.tolist()
        
        # Remove duplicate row if exists
        df_feat = df_feat.drop_duplicates()
        if self.verbose:
            print(f"✅ Removed {len(df) - len(df_feat)} duplicate rows")
        
        # ====================================================================
        # 1. DOMAIN-SPECIFIC FEATURES
        # ====================================================================
        if self.verbose:
            print("\n📊 Creating Domain-Specific Features...")
        
        # BMI Categories (WHO Classification)
        df_feat['bmi_category'] = pd.cut(
            df_feat['bmi'],
            bins=[0, 18.5, 25, 30, 35, 40, 100],
            labels=['Underweight', 'Normal', 'Overweight', 
                   'Obese_I', 'Obese_II', 'Obese_III']
        )
        
        # Simplified BMI risk
        df_feat['bmi_risk'] = df_feat['bmi_category'].map({
            'Underweight': 1,
            'Normal': 0,
            'Overweight': 1,
            'Obese_I': 2,
            'Obese_II': 3,
            'Obese_III': 4
        })
        
        # Age Groups (Insurance Industry Standard)
        df_feat['age_group'] = pd.cut(
            df_feat['age'],
            bins=[17, 25, 35, 45, 55, 65],
            labels=['18-25', '26-35', '36-45', '46-55', '56-64']
        )
        
        # Age risk score
        df_feat['age_risk'] = pd.cut(
            df_feat['age'],
            bins=[17, 30, 40, 50, 60, 65],
            labels=[1, 2, 3, 4, 5]
        ).astype(int)
        
        # Family size categories
        df_feat['family_size'] = df_feat['children'].map({
            0: 'single',
            1: 'small_family',
            2: 'small_family',
            3: 'large_family',
            4: 'large_family',
            5: 'large_family'
        })
        
        # ====================================================================
        # 2. INTERACTION FEATURES
        # ====================================================================
        if self.verbose:
            print("🔄 Creating Interaction Features...")
        
        # Critical interaction: Smoker × BMI
        df_feat['smoker_bmi'] = (df_feat['smoker'] == 'yes').astype(int) * df_feat['bmi']
        
        # Smoker × Age
        df_feat['smoker_age'] = (df_feat['smoker'] == 'yes').astype(int) * df_feat['age']
        
        # Age × BMI
        df_feat['age_bmi'] = df_feat['age'] * df_feat['bmi']
        
        # Age × Children
        df_feat['age_children'] = df_feat['age'] * df_feat['children']
        
        # BMI × Children
        df_feat['bmi_children'] = df_feat['bmi'] * df_feat['children']
        
        # High-risk indicator (smoker with high BMI)
        df_feat['high_risk'] = ((df_feat['smoker'] == 'yes') & 
                                (df_feat['bmi'] > 30)).astype(int)
        
        # ====================================================================
        # 3. POLYNOMIAL FEATURES
        # ====================================================================
        if self.verbose:
            print("📐 Creating Polynomial Features...")
        
        # Age polynomials (non-linear relationship observed)
        df_feat['age_squared'] = df_feat['age'] ** 2
        df_feat['age_cubed'] = df_feat['age'] ** 3
        df_feat['age_sqrt'] = np.sqrt(df_feat['age'])
        
        # BMI polynomials
        df_feat['bmi_squared'] = df_feat['bmi'] ** 2
        df_feat['bmi_cubed'] = df_feat['bmi'] ** 3
        df_feat['bmi_sqrt'] = np.sqrt(df_feat['bmi'])
        
        # ====================================================================
        # 4. LOGARITHMIC TRANSFORMATIONS
        # ====================================================================
        if self.verbose:
            print("📈 Creating Logarithmic Features...")
        
        df_feat['log_age'] = np.log1p(df_feat['age'])
        df_feat['log_bmi'] = np.log1p(df_feat['bmi'])
        df_feat['log_age_bmi'] = np.log1p(df_feat['age'] * df_feat['bmi'])
        
        # ====================================================================
        # 5. STATISTICAL FEATURES
        # ====================================================================
        if self.verbose:
            print("📊 Creating Statistical Features...")
        
        # Z-scores (standardized features)
        df_feat['age_zscore'] = (df_feat['age'] - df_feat['age'].mean()) / df_feat['age'].std()
        df_feat['bmi_zscore'] = (df_feat['bmi'] - df_feat['bmi'].mean()) / df_feat['bmi'].std()
        
        # Percentile ranks
        df_feat['age_percentile'] = df_feat['age'].rank(pct=True)
        df_feat['bmi_percentile'] = df_feat['bmi'].rank(pct=True)
        
        # ====================================================================
        # 6. COMPOSITE RISK SCORES
        # ====================================================================
        if self.verbose:
            print("🎯 Creating Composite Risk Scores...")
        
        # Overall risk score (weighted combination)
        df_feat['risk_score'] = (
            df_feat['age_risk'] * 0.25 +
            df_feat['bmi_risk'] * 0.20 +
            (df_feat['smoker'] == 'yes').astype(int) * 5 +
            df_feat['children'] * 0.1
        )
        
        # # Health score (inverse of risk factors)
        # df_feat['health_score'] = (
        #     (100 - df_feat['age']) * 0.3 +
        #     (50 - abs(df_feat['bmi'] - 22)) * 0.3 +  # 22 is optimal BMI
        #     (df_feat['smoker'] == 'no').astype(int) * 40
        # )
        
        # # ====================================================================
        # # 7. REGIONAL ENCODING
        # # ====================================================================
        # if self.verbose:
        #     print("🗺️ Creating Regional Features...")
        
        # # Regional risk factors (based on healthcare costs by region)
        # region_risk_map = {
        #     'southeast': 1.2,
        #     'southwest': 1.0,
        #     'northeast': 1.1,
        #     'northwest': 0.9
        # }
        # df_feat['region_risk_factor'] = df_feat['region'].map(region_risk_map)
        
        # ====================================================================
        # 8. BINNING CONTINUOUS FEATURES
        # ====================================================================
        if self.verbose:
            print("📦 Creating Binned Features...")
        
        # BMI bins
        df_feat['bmi_bin_5'] = pd.cut(df_feat['bmi'], bins=5, labels=False)
        df_feat['bmi_bin_10'] = pd.cut(df_feat['bmi'], bins=10, labels=False)
        
        # Age bins
        df_feat['age_bin_5'] = pd.cut(df_feat['age'], bins=5, labels=False)
        df_feat['age_bin_10'] = pd.cut(df_feat['age'], bins=10, labels=False)
        
        # ====================================================================
        # FEATURE SUMMARY
        # ====================================================================
        new_features = [col for col in df_feat.columns if col not in self.original_features]
        self.feature_names = new_features
        
        if self.verbose:
            print("\n" + "="*80)
            print(f"✅ FEATURE ENGINEERING COMPLETE!")
            print(f"📊 Original features: {len(self.original_features)}")
            print(f"📊 New features created: {len(new_features)}")
            print(f"📊 Total features: {len(df_feat.columns)}")
            print("="*80)
        
        return df_feat
    
    def get_feature_names(self):
        """Return list of created feature names"""
        return self.feature_names

# Apply feature engineering
engineer = AdvancedFeatureEngineer(verbose=True)
df_engineered = engineer.fit_transform(df)

# Display sample of engineered features
print("\n📋 Sample of Engineered Dataset:")
display(df_engineered.head())

# Show new features created
print("\n📝 New Features Created:")
for i, feature in enumerate(engineer.get_feature_names(), 1):
    print(f"  {i:2d}. {feature}")

🔧 Starting Feature Engineering Pipeline...
✅ Removed 1 duplicate rows

📊 Creating Domain-Specific Features...
🔄 Creating Interaction Features...
📐 Creating Polynomial Features...
📈 Creating Logarithmic Features...
📊 Creating Statistical Features...
🎯 Creating Composite Risk Scores...
📦 Creating Binned Features...

✅ FEATURE ENGINEERING COMPLETE!
📊 Original features: 7
📊 New features created: 29
📊 Total features: 36

📋 Sample of Engineered Dataset:


,age,sex,bmi,children,smoker,region,charges,bmi_category,bmi_risk,age_group,...,log_age_bmi,age_zscore,bmi_zscore,age_percentile,bmi_percentile,risk_score,bmi_bin_5,bmi_bin_10,age_bin_5,age_bin_10
0,19,female,27.900,0,yes,southwest,16884.92400,Overweight,1,18-25,...,6.274950,-1.439879,-0.452990,0.077038,0.343306,5.45,1,3,0,0
1,18,male,33.770,1,no,southeast,1725.55230,Obese_I,2,18-25,...,6.411588,-1.511082,0.509231,0.026178,0.704936,0.75,2,4,0,0
2,28,male,33.000,3,no,southeast,4449.46200,Obese_I,2,26-35,...,6.829794,-0.799051,0.383011,0.280853,0.657068,0.95,2,4,1,2
3,33,male,22.705,0,no,northwest,21984.47061,Normal,0,26-35,...,6.620426,-0.443036,-1.304564,0.381077,0.092745,0.50,0,1,1,3
4,32,male,28.880,0,no,northwest,3866.85520,Overweight,1,26-35,...,6.829967,-0.514239,-0.292347,0.361631,0.405011,0.70,1,3,1,3



📝 New Features Created:
   1. bmi_category
   2. bmi_risk
   3. age_group
   4. age_risk
   5. family_size
   6. smoker_bmi
   7. smoker_age
   8. age_bmi
   9. age_children
  10. bmi_children
  11. high_risk
  12. age_squared
  13. age_cubed
  14. age_sqrt
  15. bmi_squared
  16. bmi_cubed
  17. bmi_sqrt
  18. log_age
  19. log_bmi
  20. log_age_bmi
  21. age_zscore
  22. bmi_zscore
  23. age_percentile
  24. bmi_percentile
  25. risk_score
  26. bmi_bin_5
  27. bmi_bin_10
  28. age_bin_5
  29. age_bin_10


In [4]:
X = df_engineered.drop(['charges'],axis=1)
Y = df_engineered['charges']
categorical_cols = X.select_dtypes(include=['object','category']).columns
numerical_cols = X.select_dtypes(include=[np.number]).columns
print(f"📊 Categorical columns: {list(categorical_cols)}")
print(f"📊 Numerical columns: {len(numerical_cols)} columns")

from sklearn.preprocessing import LabelEncoder

X_encoded = X.copy()
label_encoders = {}

for col in X_encoded:
    le = LabelEncoder()
    X_encoded[col] = le.fit_transform(X_encoded[col])



📊 Categorical columns: ['sex', 'smoker', 'region', 'bmi_category', 'age_group', 'family_size']
📊 Numerical columns: 29 columns


In [5]:
# ============================================================================
# FEATURE SELECTION AND IMPORTANCE ANALYSIS 
# ============================================================================

X = df_engineered.drop(['charges'], axis=1)
Y = df_engineered['charges']

categorical_cols = X.select_dtypes(include=['object', 'category']).columns
numerical_cols = X.select_dtypes(include=[np.number]).columns


print(f"📊 Categorical columns: {list(categorical_cols)}")
print(f"📊 Numerical columns: {len(numerical_cols)} columns")

# Encode ALL categorical variables
from sklearn.preprocessing import LabelEncoder

X_encoded = X.copy()
label_encoders = {}

# Encode each categorical column
for col in categorical_cols:
    le = LabelEncoder()
    
    # Handle different column types
    if X_encoded[col].dtype.name == 'category':
        # For categorical columns, convert to string first
        X_encoded[col] = X_encoded[col].astype(str)
    
    # Replace NaN with 'missing' string (now safe for all column types)
    X_encoded[col] = X_encoded[col].fillna('missing')
    
    # Encode the column
    X_encoded[col] = le.fit_transform(X_encoded[col].astype(str))
    label_encoders[col] = le

print(f"\n✅ Encoded {len(categorical_cols)} categorical columns")

# Verify all columns are numeric now
non_numeric = X_encoded.select_dtypes(exclude=[np.number]).columns
if len(non_numeric) > 0:
    print(f"⚠️ Warning: Still have non-numeric columns: {list(non_numeric)}")
    # Force encode any remaining non-numeric columns
    for col in non_numeric:
        le = LabelEncoder()
        X_encoded[col] = le.fit_transform(X_encoded[col].astype(str))
        label_encoders[col] = le
else:
    print("✅ All columns are now numeric")

# ====================================================================
# MUTUAL INFORMATION SCORES
# ====================================================================
print("\n🔍 Calculating Mutual Information Scores...")

# Now this should work without errors
mi_scores = mutual_info_regression(X_encoded, y, random_state=RANDOM_STATE)
mi_scores_df = pd.DataFrame({
    'Feature': X_encoded.columns,
    'MI Score': mi_scores
}).sort_values('MI Score', ascending=False)

# Display top 20 features
print("\n📊 Top 20 Features by Mutual Information:")
display(mi_scores_df.head(20))

# Visualize feature importance
plt.figure(figsize=(12, 8))
top_features = mi_scores_df.head(20)
plt.barh(range(len(top_features)), top_features['MI Score'].values)
plt.yticks(range(len(top_features)), top_features['Feature'].values)
plt.xlabel('Mutual Information Score')
plt.title('Top 20 Features by Mutual Information Score')
plt.tight_layout()
plt.show()

# ====================================================================
# RANDOM FOREST FEATURE IMPORTANCE
# ====================================================================
print("\n🌲 Training Random Forest for Feature Importance...")

rf_model = RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
rf_model.fit(X_encoded, y)

# Get feature importances
rf_importance_df = pd.DataFrame({
    'Feature': X_encoded.columns,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("\n📊 Top 20 Features by Random Forest Importance:")
display(rf_importance_df.head(20))

# ====================================================================
# SELECT FINAL FEATURES
# ====================================================================
print("\n🎯 Selecting Final Features...")

# Combine both importance measures
importance_combined = pd.merge(
    mi_scores_df, 
    rf_importance_df, 
    on='Feature'
)

# Normalize scores to 0-1 range
importance_combined['MI_normalized'] = (importance_combined['MI Score'] / 
                                        importance_combined['MI Score'].max())
importance_combined['RF_normalized'] = (importance_combined['Importance'] / 
                                        importance_combined['Importance'].max())

# Combined score (average of both)
importance_combined['Combined_Score'] = (importance_combined['MI_normalized'] + 
                                         importance_combined['RF_normalized']) / 2

importance_combined = importance_combined.sort_values('Combined_Score', ascending=False)

print("\n📊 Top 20 Features by Combined Importance:")
display(importance_combined[['Feature', 'Combined_Score']].head(20))

# Select features above threshold
threshold = 0.1  # Select features with combined score > 0.1
selected_features = importance_combined[importance_combined['Combined_Score'] > threshold]['Feature'].tolist()

# Always include original important features
must_have_features = ['age', 'bmi', 'children', 'smoker', 'sex', 'region']
for feature in must_have_features:
    if feature not in selected_features:
        selected_features.append(feature)

print(f"\n✅ Selected {len(selected_features)} features for modeling")
print(f"📝 Selected features: {selected_features}")

# Create final feature set
X_selected = X_encoded[selected_features]

print("\n" + "="*80)
print("✅ FEATURE ENGINEERING & SELECTION COMPLETE!")
print(f"📊 Final feature set: {X_selected.shape}")
print("="*80)

📊 Categorical columns: ['sex', 'smoker', 'region', 'bmi_category', 'age_group', 'family_size']
📊 Numerical columns: 29 columns

✅ Encoded 6 categorical columns
✅ All columns are now numeric

🔍 Calculating Mutual Information Scores...


NameError: name 'y' is not defined